# GeoSlide — Phase 3: Data Preprocessing

## Introduction

This notebook implements **Phase 3 (Data Preprocessing)** of the GeoSlide project, a landslide
early-warning system built on Wireless Sensor Network (WSN) data.

**What is data preprocessing?**
Data preprocessing is the process of transforming raw, real-world data into a clean, consistent,
and well-structured form that is ready to be consumed by downstream analytical or modeling steps.
Raw sensor data typically contains issues such as missing readings, duplicate log entries,
inconsistent data types, and features that live on very different numerical scales. Left
unaddressed, these issues can bias models, produce misleading results, or simply cause errors.

**Objectives of this notebook**
1. Load the raw WSN landslide dataset.
2. Verify the quality of the dataset (missing values, duplicates, data types).
3. Separate the dataset into features (`X`) and target (`y`).
4. Identify and encode any categorical features, if present.
5. Split the dataset into training and testing subsets (80/20), preserving class balance
   via stratification.
6. Scale numerical features using `StandardScaler`, fit **only** on the training data to
   avoid data leakage.
7. Verify the shapes and class distributions of the resulting splits.
8. Summarize all preprocessing steps performed.

**Scope note:** This notebook is strictly limited to preprocessing. It does **not** perform
feature engineering (creating new derived features), does **not** train any machine learning
models, and does **not** perform model evaluation. Those activities belong to later phases of
the GeoSlide project.


## 2. Import Libraries

We import only the libraries required for preprocessing:

- `pandas` and `numpy` for data loading and manipulation.
- `train_test_split` from `sklearn.model_selection` to create the train/test partitions.
- `StandardScaler` from `sklearn.preprocessing` to standardize numerical features.
- `warnings` to keep the notebook output clean and readable.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
print(f"pandas version : {pd.__version__}")
print(f"numpy version  : {np.__version__}")


Libraries imported successfully.
pandas version : 2.2.2
numpy version  : 2.0.2


## 3. Load Dataset

We load the raw WSN landslide dataset from:

```
datasets/training/wsn_landslide_data.csv
```

After loading, we inspect the shape of the dataset (number of rows and columns) and preview
the first few records to get an initial sense of the data.


In [2]:
DATA_PATH = "/content/wsn_landslide_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


Dataset shape: 9864 rows x 35 columns


,Rainfall_mm,Slope_Angle,Soil_Saturation,Vegetation_Cover,Rainfall_3Day,Rainfall_7Day,Aspect,Elevation_m,NDVI_Index,Land_Use_Urban,...,Soil_Type_Silt,Soil_Type_Clay,Pore_Water_Pressure_kPa,Soil_Moisture_Content,Microseismic_Activity,Acoustic_Emission_dB,Soil_Strain,Soil_Temperature_C,TDR_Reflection_Index,Label
0,164.695364,59.783332,0.821479,0.107339,260.138381,79.297169,346.674199,733.776448,0.191948,1,...,1,0,133.943194,0.143732,0.290945,51.021834,0.005167,22.760036,0.799847,1
1,34.908086,15.153084,0.100428,0.960150,510.295547,247.923576,104.462371,467.708643,0.798321,1,...,0,1,90.788608,0.266484,0.651758,39.837282,0.003443,15.558373,1.181071,0
2,38.761727,13.135384,0.286526,0.833608,297.730266,194.327012,336.671287,1880.826807,0.479456,1,...,0,1,83.041150,0.129426,0.440714,68.902366,0.009999,6.205760,1.184971,0
3,32.199977,10.674734,0.255230,0.847569,231.640610,295.139546,300.742864,964.080336,-0.084314,1,...,1,0,196.089305,0.240198,0.794001,80.196960,0.003850,25.486545,0.677944,0
4,218.114032,48.839269,0.720071,0.018383,330.278249,301.288824,155.550502,165.699102,0.810869,0,...,1,1,106.778890,0.345724,0.009160,99.919786,0.003061,7.270319,0.882642,1


**Column overview** — a quick look at the column names and their inferred data types.

In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Rainfall_mm                 9864 non-null   float64
 1   Slope_Angle                 9864 non-null   float64
 2   Soil_Saturation             9864 non-null   float64
 3   Vegetation_Cover            9864 non-null   float64
 4   Rainfall_3Day               9864 non-null   float64
 5   Rainfall_7Day               9864 non-null   float64
 6   Aspect                      9864 non-null   float64
 7   Elevation_m                 9864 non-null   float64
 8   NDVI_Index                  9864 non-null   float64
 9   Land_Use_Urban              9864 non-null   int64  
 10  Land_Use_Forest             9864 non-null   int64  
 11  Land_Use_Agriculture        9864 non-null   int64  
 12  Earthquake_Activity         9864 non-null   float64
 13  Proximity_to_Water          9864 

## 4. Verify Dataset Quality

Before doing anything else, we need to confirm that the dataset is reliable enough to work
with. We check three things:

1. **Missing values** — Are there any null/NaN entries that need to be handled?
2. **Duplicate records** — Are there any fully duplicated rows that should be removed to
   avoid biasing the train/test split?
3. **Data types** — Are numerical columns actually stored as numeric types, and are there
   any unexpected categorical/text columns?


In [4]:
# 4.1 Missing values per column
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct.round(2)
}).sort_values("missing_count", ascending=False)

print("Missing values per column:")
missing_summary


Missing values per column:


,missing_count,missing_pct
Rainfall_mm,0,0.0
Slope_Angle,0,0.0
Soil_Saturation,0,0.0
Vegetation_Cover,0,0.0
Rainfall_3Day,0,0.0
Rainfall_7Day,0,0.0
Aspect,0,0.0
Elevation_m,0,0.0
NDVI_Index,0,0.0
Land_Use_Urban,0,0.0


In [5]:
# 4.2 Duplicate records
n_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows: {n_duplicates}")

if n_duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicate rows found. No action needed.")


Number of fully duplicated rows: 0
No duplicate rows found. No action needed.


In [6]:
# 4.3 Data types
print("Data types of each column:")
print(df.dtypes)


Data types of each column:
Rainfall_mm                   float64
Slope_Angle                   float64
Soil_Saturation               float64
Vegetation_Cover              float64
Rainfall_3Day                 float64
Rainfall_7Day                 float64
Aspect                        float64
Elevation_m                   float64
NDVI_Index                    float64
Land_Use_Urban                  int64
Land_Use_Forest                 int64
Land_Use_Agriculture            int64
Earthquake_Activity           float64
Proximity_to_Water            float64
Distance_to_Road_m            float64
Temperature_C                 float64
Humidity_percent              float64
Soil_pH                       float64
Clay_Content                  float64
Sand_Content                  float64
Silt_Content                  float64
Soil_Erosion_Rate             float64
Historical_Landslide_Count      int64
Soil_Type_Gravel                int64
Soil_Type_Sand                  int64
Soil_Type_Silt         

**Findings**

- The missing-value summary above shows, per column, how many entries (and what percentage of
  the dataset) are missing. If any columns show missing values, they should be investigated and
  handled (e.g., imputation or removal) before the dataset is used downstream; if the dataset is
  complete, no imputation is required.
- Any fully duplicated rows were identified and removed above so that the same sensor reading
  does not appear in both the training and testing sets.
- The data-type check confirms whether each column is numeric (`int64`/`float64`) as expected
  for sensor readings, or whether any column is stored as `object` (text), which would need to
  be addressed in the categorical-feature step below.


## 5. Separate Features and Target

We now split the dataframe into:

- `X` — the input features (all sensor/environmental readings used to make a prediction).
- `y` — the target variable that we want to predict (the landslide occurrence/label column).

We assume the target column is named `label` (a common convention for this type of dataset).
If your dataset uses a different column name for the target, update `TARGET_COLUMN` below
accordingly.


In [9]:
TARGET_COLUMN = "Label"  # update this if the target column has a different name

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' not found in dataset columns: {list(df.columns)}. "
        "Please update TARGET_COLUMN to match the actual target column name."
    )

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nFeature columns:")
print(list(X.columns))

X shape: (9864, 34)
y shape: (9864,)

Feature columns:
['Rainfall_mm', 'Slope_Angle', 'Soil_Saturation', 'Vegetation_Cover', 'Rainfall_3Day', 'Rainfall_7Day', 'Aspect', 'Elevation_m', 'NDVI_Index', 'Land_Use_Urban', 'Land_Use_Forest', 'Land_Use_Agriculture', 'Earthquake_Activity', 'Proximity_to_Water', 'Distance_to_Road_m', 'Temperature_C', 'Humidity_percent', 'Soil_pH', 'Clay_Content', 'Sand_Content', 'Silt_Content', 'Soil_Erosion_Rate', 'Historical_Landslide_Count', 'Soil_Type_Gravel', 'Soil_Type_Sand', 'Soil_Type_Silt', 'Soil_Type_Clay', 'Pore_Water_Pressure_kPa', 'Soil_Moisture_Content', 'Microseismic_Activity', 'Acoustic_Emission_dB', 'Soil_Strain', 'Soil_Temperature_C', 'TDR_Reflection_Index']


## 6. Check for Categorical Features

We inspect `X` for any non-numeric (categorical/text) columns. WSN sensor data is typically
composed entirely of numerical readings (e.g., soil moisture, rainfall, vibration, temperature),
so we expect few or no categorical columns. If categorical columns are found, we encode them
using one-hot encoding so that they can be used in downstream numerical computations. If no
categorical columns are found, no encoding is necessary and this step is skipped.


In [10]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Categorical columns found: {categorical_cols}")

if len(categorical_cols) > 0:
    print("\nEncoding categorical columns using one-hot encoding...")
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f"X shape after encoding: {X.shape}")
else:
    print("\nNo categorical columns detected. All features are already numeric, "
          "so encoding is unnecessary at this stage.")


Categorical columns found: []

No categorical columns detected. All features are already numeric, so encoding is unnecessary at this stage.


## 7. Train-Test Split

We split the dataset into training (80%) and testing (20%) subsets using
`train_test_split`. Key choices:

- `test_size=0.2` — reserves 20% of the data for testing.
- `random_state=42` — ensures the split is reproducible.
- `stratify=y` — preserves the original class proportions of the target variable in both the
  training and testing sets, which is especially important for imbalanced classification
  problems such as landslide occurrence prediction.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")


X_train shape: (7891, 34)
X_test shape : (1973, 34)
y_train shape: (7891,)
y_test shape : (1973,)


## 8. Feature Scaling

Many features (e.g., rainfall in mm, soil moisture in %, vibration sensor units) live on very
different numerical scales. `StandardScaler` standardizes each feature to have a mean of 0 and
a standard deviation of 1.

**Important — avoiding data leakage:** The scaler is `fit` **only** on `X_train`. It is then
used to `transform` both `X_train` and `X_test`. Fitting on the full dataset (or on the test
set) would leak information about the test distribution into training, which would produce an
overly optimistic and invalid evaluation later on. The target variable `y` is never scaled,
since it represents class labels, not a continuous numerical feature.


In [12]:
scaler = StandardScaler()

# Fit only on training data
scaler.fit(X_train)

# Transform both train and test sets using the same fitted scaler
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Feature scaling complete.")
print("\nScaled X_train preview:")
X_train_scaled.head()


Feature scaling complete.

Scaled X_train preview:


,Rainfall_mm,Slope_Angle,Soil_Saturation,Vegetation_Cover,Rainfall_3Day,Rainfall_7Day,Aspect,Elevation_m,NDVI_Index,Land_Use_Urban,...,Soil_Type_Sand,Soil_Type_Silt,Soil_Type_Clay,Pore_Water_Pressure_kPa,Soil_Moisture_Content,Microseismic_Activity,Acoustic_Emission_dB,Soil_Strain,Soil_Temperature_C,TDR_Reflection_Index
8634,-0.776612,-0.727370,-0.840921,0.385417,-1.209486,-0.698680,1.390974,1.502746,0.128415,0.99054,...,1.001395,-1.003682,-1.000634,-1.096011,1.359494,0.651683,1.670043,1.101824,0.478977,0.688241
7228,1.351268,1.060713,1.078518,-1.365724,0.327845,-0.095339,1.237861,-0.726864,-1.278402,0.99054,...,1.001395,-1.003682,0.999367,-0.665429,-0.319344,1.619017,0.050107,-0.432909,-0.216265,-1.390059
6455,0.476365,0.194739,0.598754,-0.622378,0.578487,1.024692,-1.229386,-0.939964,0.188844,0.99054,...,-0.998607,-1.003682,-1.000634,-0.288374,-0.078339,-1.221620,0.115438,-0.297767,0.082348,-0.309471
3995,0.670669,1.656943,0.852604,-0.911587,0.875359,-0.749202,1.408954,-0.799226,-0.119543,0.99054,...,-0.998607,-1.003682,0.999367,-1.360326,-1.437382,-1.085371,0.194116,0.583525,-1.277326,0.730364
5480,-1.092058,-0.730377,-0.717082,0.387605,-1.486321,-1.751035,-0.777141,1.573934,-1.361991,0.99054,...,1.001395,0.996332,0.999367,-1.427561,0.107011,-1.725073,1.532308,-0.777628,1.209556,0.684212


## 9. Verify Results

Finally, we verify the outcome of the preprocessing pipeline by confirming the shapes of all
resulting arrays/dataframes and checking that the class distribution of the target variable is
consistent (proportionally) between the training and testing sets.


In [13]:
print("Final shapes:")
print(f"  X_train_scaled: {X_train_scaled.shape}")
print(f"  X_test_scaled : {X_test_scaled.shape}")
print(f"  y_train       : {y_train.shape}")
print(f"  y_test        : {y_test.shape}")


Final shapes:
  X_train_scaled: (7891, 34)
  X_test_scaled : (1973, 34)
  y_train       : (7891,)
  y_test        : (1973,)


In [14]:
print("Class distribution in y_train (proportion):")
print(y_train.value_counts(normalize=True).round(4))

print("\nClass distribution in y_test (proportion):")
print(y_test.value_counts(normalize=True).round(4))

print("\nClass distribution in y_train (counts):")
print(y_train.value_counts())

print("\nClass distribution in y_test (counts):")
print(y_test.value_counts())


Class distribution in y_train (proportion):
Label
1    0.5027
0    0.4973
Name: proportion, dtype: float64

Class distribution in y_test (proportion):
Label
1    0.5028
0    0.4972
Name: proportion, dtype: float64

Class distribution in y_train (counts):
Label
1    3967
0    3924
Name: count, dtype: int64

Class distribution in y_test (counts):
Label
1    992
0    981
Name: count, dtype: int64


If the class proportions in `y_train` and `y_test` are approximately equal, the stratified
split successfully preserved the original class balance across both subsets.


## Preprocessing Summary

This notebook completed Phase 3 (Data Preprocessing) of the GeoSlide project. The following
steps were performed, in order:

1. **Loaded** the raw WSN landslide dataset from `datasets/training/wsn_landslide_data.csv` and
   inspected its shape and structure.
2. **Verified data quality** by checking for missing values, duplicate records, and correctness
   of data types. Duplicate rows, if any, were removed.
3. **Separated** the dataset into feature matrix `X` and target vector `y`.
4. **Checked for categorical features** in `X` and applied one-hot encoding where necessary; if
   no categorical columns were present, this step was skipped as unnecessary.
5. **Split** the data into training (80%) and testing (20%) subsets using
   `train_test_split(..., random_state=42, stratify=y)` to ensure reproducibility and preserve
   class balance.
6. **Scaled** numerical features using `StandardScaler`, fitting the scaler only on `X_train`
   and applying the same transformation to `X_test`, to prevent data leakage. The target
   variable `y` was left unscaled.
7. **Verified** the final shapes of `X_train`, `X_test`, `y_train`, and `y_test`, and confirmed
   that class proportions were preserved across the training and testing splits.

**Outputs available for the next phase:**
- `X_train_scaled`, `X_test_scaled` — scaled feature sets ready for modeling.
- `y_train`, `y_test` — corresponding target labels.
- `scaler` — the fitted `StandardScaler` instance (should be reused, not refit, on any future
  data such as a validation set).

**Out of scope (explicitly not performed in this notebook):**
- Feature engineering (creation of new/derived features).
- Model training of any kind.
- Model evaluation or performance metrics.

These activities belong to subsequent phases of the GeoSlide project pipeline.
